## Breaking Down Tasks with Prompt Chaining

Welcome! In our previous lesson, you learned the fundamentals of communicating with Claude through the Anthropic API. You mastered sending single requests, understanding response structures, and managing multi-turn conversations within a single interaction. Now, you’re ready to take the next step and unlock even more powerful ways to work with Claude.

In this lesson, you’ll discover how to design and implement multi-step workflows using Claude, with a special focus on a technique called prompt chaining. By the end, you’ll know how to break down sophisticated tasks into manageable, reliable steps and connect them together for robust AI-powered solutions.
What is a Workflow?

A workflow is a structured sequence of steps or actions designed to accomplish a specific goal. In the context of AI systems, workflows help you organize and coordinate tasks so that each step has a clear purpose, defined inputs and outputs, and measurable success criteria. There are many types of workflows—some involve a single interaction, while others may require multiple steps, validation, or branching logic. Well-designed workflows make complex processes more predictable, easier to debug, and simpler to maintain.
Prompt Chaining and Why it Matters

Prompt chaining is one specific workflow pattern where you connect multiple separate Claude calls together, with each call building upon the output of the previous one. Unlike multi-turn conversations that happen within a single session, prompt chaining involves distinct API calls that work together to solve complex problems step by step.

The power of prompt chaining lies in its reliability and modularity. Instead of asking Claude to perform multiple complex tasks in a single prompt (which can lead to inconsistent results), you break the work into focused steps where you can validate and control the output at each stage. This approach makes your AI workflows more predictable and easier to debug.
Design the Workflow Before Coding

Before writing any code, it's important to break your task into clear, manageable steps. For our example, we'll build a simple three-step workflow:

    Generate a summary about AI in healthcare, with a strict character limit (around 300 characters).
    Validate that the summary meets the character requirement.
    Translate the validated summary into Spanish, returning only the translated text.

Each step will have its own focused prompt and clear input/output, making the workflow easy to follow and debug. This approach helps ensure each part works as expected before moving to the next.
Step 1: Generate a Constrained Summary

Let's start building our chain by creating the first step: generating a summary with specific character constraints. This step demonstrates how to use system prompts and user messages effectively to get predictable output from Claude.

```Python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)
```

Notice how we separate the system prompt from the user message. The system prompt establishes Claude's role as a summary writer, while the user message contains the specific task and constraints. This separation makes our prompts more maintainable and allows us to reuse the same system prompt for different summary tasks.

The user message is explicit about the character requirement. Instead of saying "write a short summary", we specify exactly "300 characters" to make the constraint testable and clear. This precision is essential in prompt chaining because the output of this step becomes the input for the next step.

When you run this code, you'll see output similar to:

Plain text
Summary:
AI in healthcare revolutionizes diagnosis through medical imaging analysis, enables personalized treatment plans, automates administrative tasks, assists in drug discovery, and powers virtual health assistants for patient care and monitoring.

The summary_response.content[0].text extraction pattern should be familiar from our previous lesson. We're accessing the first content block and extracting its text content, which works well for simple text responses like this summary.
Step 2: Validate and Guardrail the Output

The second step in our chain adds a crucial validation layer that ensures our summary meets the character requirements before proceeding to translation. This validation step demonstrates how to build reliable guardrails into your prompt chains.

```Python
# Step 2: Validate that the summary meets our character requirements
if not (250 <= len(summary_text) <= 350):
    raise ValueError(f"Summary does not meet character requirement (250-350 characters). Got {len(summary_text)} characters.")

print(f"✅ SUCCESS: Summary meets character requirement: {len(summary_text)} characters")
```

This validation step uses a programmatic check rather than asking Claude to validate its own output. We define a reasonable range (250-350 characters) instead of requiring exactly 300 characters, which gives Claude some flexibility while still meeting our needs.

When the validation fails, we raise a ValueError with a descriptive message that includes both the expected range and the actual character count. This makes debugging easier when your chain encounters problems. In production systems, you might want to implement retry logic here, perhaps asking Claude to revise the summary with tighter constraints.

When the validation passes, you'll see output like:

Plain text
✅ SUCCESS: Summary meets character requirement: 298 characters

Without validation, a summary that's too long or too short could cause problems in subsequent steps. By catching and handling constraint violations early, you make your entire workflow more robust.
Step 3: Feed the Output into Translation

The third step demonstrates the core concept of prompt chaining: using the output from one Claude call as input to another. This step takes our validated summary and translates it into Spanish using a focused system prompt.

```Python
# Step 3: Chain the summary output as input to a translation task
translation_prompt = "You are a professional translator that provides accurate Spanish translations."

translation_messages = [
    {
        "role": "user",
        "content": f"Return me just the Spanish translation of the following text:\n\n{summary_text}"
    }
]

# Send the second request to Claude using the summary from the first call
translation_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=translation_messages,
    system=translation_prompt
)

# Extract and display the final translation result
translation_text = translation_response.content[0].text
print("Spanish Translation:")
print(translation_text)
```

The system prompt for this step is focused specifically on translation rather than general assistance. This specialization helps Claude understand its role in this step of the chain and produces more consistent results.

The key insight here is how we safely pass the summary_text variable from step one into the user message for step three. We use an f-string to embed the summary directly into the prompt, creating a clear separation between our instruction ("Return me just the Spanish translation") and the content to be translated.

When you run this final step, you'll see output like:

Plain text
Spanish Translation:
La IA en salud revoluciona el diagnóstico mediante análisis de imágenes médicas, permite planes de tratamiento personalizados, automatiza tareas administrativas, ayuda en el descubrimiento de medicamentos y potencia asistentes virtuales de salud para el cuidado y monitoreo de pacientes.

Each step builds naturally on the previous one, creating a smooth workflow from English summary generation through validation to Spanish translation.
Summary & Prep for Practice

You've successfully designed and implemented a three-step prompt chain that demonstrates the core concepts of sequential AI workflows. Your chain writes a constrained summary, validates that it meets requirements, then uses that validated output as input for translation. The key patterns you've learned include decomposing complex tasks into focused steps, using validation and guardrails between steps, and safely passing output from one Claude call as input to the next.

In the upcoming practice exercises, you'll implement this code yourself and extend it with additional features. The foundation you've built with prompt chaining opens up possibilities for much more sophisticated AI workflows. As you continue through this course, you'll see how these basic chaining concepts extend to tool usage, dynamic workflows, and complex agent behaviors that can handle real-world business problems.


## Building Your First Chain Link

Now that you understand the power of prompt chaining from the lesson, it's time to build your first chain link! In this exercise, you'll implement the foundation of a prompt chain by creating the summary generation step, which will later feed into validation and translation.

Your goal is to complete the missing code that makes a successful API call to Claude for generating a constrained summary. You'll need to:

    Create a system prompt that establishes Claude's role as a summary assistant
    Build a user message requesting a 300-character summary about something
    Make the complete API call with all required parameters
    Extract the summary text from Claude's response

Pay attention to the format of the messages list, and remember to include both the system parameter and the messages parameter in your API call.

Once you complete this step, you'll have created the first reliable link in a prompt chain that produces focused, constrained output ready for the next step in the workflow!

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
# TODO: Create a system prompt that tells Claude to be a helpful summary assistant
summary_prompt = 

# TODO: Create the messages list with a user message requesting a 300-character summary about something
summary_messages = 

# TODO: Make the API call to Claude with model, max_tokens=2000, messages, and system parameters
summary_response = 

# TODO: Extract the text from the response and assign it to summary_text
summary_text = 

print("Summary:")
print(summary_text)

```

To build the first link in your prompt chain, you need to use the Anthropic Python SDK to configure the system prompt, format the messages array, send the request, and safely extract the generated text.

Here is the complete, working Python code with all the missing pieces implemented:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
# TODO: Create a system prompt that tells Claude to be a helpful summary assistant
summary_prompt = "You are a helpful summary assistant. Your task is to extract the absolute core ideas of the text provided and write a highly condensed summary."

# TODO: Create the messages list with a user message requesting a 300-character summary about something
summary_messages = [
    {
        "role": "user",
        "content": (
            "Please write a concise summary of the concept of 'Prompt Chaining' in generative AI. "
            "The final summary must be under 300 characters long, including spaces."
        )
    }
]

# TODO: Make the API call to Claude with model, max_tokens=2000, messages, and system parameters
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    system=summary_prompt,
    messages=summary_messages
)

# TODO: Extract the text from the response and assign it to summary_text
summary_text = summary_response.content[0].text

print("Summary:")
print(summary_text)

```

### 🧱 Breaking Down the Chain Link

* **`system=summary_prompt`**: This defines the overarching guardrails and behavioral identity for Claude before it parses the input data.
* **`summary_messages`**: This passes the actual task instructions. By clearly stating constraints like *"under 300 characters long"*, you ensure that the text payload generated in this link is predictable and well-formatted for downstream automation.
* **`summary_response.content[0].text`**: This safely navigates Anthropic's block structure to capture the string response, making it ready to be stored in `summary_text` so you can pass it to the next prompt link in your application workflow.

## Adding Guardrails to Your Chain

Great work building your first chain link! Now it's time to add the crucial middle step that makes prompt chains reliable and robust.

In this exercise, you'll implement the validation layer that sits between your summary generation and translation steps. This validation acts as a quality control checkpoint, ensuring your data meets requirements before moving to the next step in the chain.

Your task is to complete the missing validation logic in Step 2:

    Add a condition that checks whether the summary length is outside the range of 250 - 350 characters
        If true, raise a descriptive ValueError when validation fails, including both the expected range and the actual character count
    Print a success message when the validation passes, showing the actual character count

The summary generation (Step 1) is already working and ready to go. Once you add the validation guardrails, you'll have a robust two-step prompt chain that generates and validates content reliably!

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our character requirements

# TODO: Add an if condition to check if the summary length is NOT between 250-350 characters
    # TODO: If true, raise a ValueError with a descriptive message that includes both the expected range and actual character count

# TODO: Print a success message showing the validation passed and the actual character count

```

To build a robust, deterministic prompt chain, adding an intermediate validation layer acts as a critical guardrail. This step prevents faulty or unconstrained outputs from propagating downstream to subsequent LLM steps.

Here is the completed script featuring your missing validation logic and error boundaries implemented cleanly in Python:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our character requirements

# Measure the actual character count of the generated text
actual_length = len(summary_text)

# Check whether the summary length is outside the acceptable range of 250 - 350 characters
if not (250 <= actual_length <= 350):
    raise ValueError(
        f"Validation Failed: Generated summary length is outside the permitted boundary. "
        f"Expected between 250 and 350 characters, but got {actual_length} characters."
    )

# Print a success message showing the validation passed and the actual character count
print(f"\n✅ Validation Passed! Summary length is exactly {actual_length} characters.")

```

### Why This Guardrail Pattern Matters:

* **Fails Fast:** If the model introduces length drift (e.g., writing a summary that is 450 characters long), the code explicitly cuts off execution immediately. This saves downstream token costs and prevents system exceptions further along your pipeline.
* **Informative Diagnostics:** Including both the boundaries ($250 \le \text{length} \le 350$) and the actual calculated metric directly inside the `ValueError` runtime exception allows you to quickly adjust your prompt system directives or configure retry loop patterns if the model drifts from constraints.

Ah, the classic text-length dilemma! This trace shows exactly why we put that validation guardrail in the middle of our prompt chain.

Even though you told Claude to write a "300 characters summary," Large Language Models don't actually count characters or words while they are generating text; they process and predict information using **tokens** (word fragments). Because of this token-to-character abstraction, getting an exact character count on the first try can be slightly erratic, pushing your output over the boundary to **362 characters**.

To fix this and make your prompt chain resilient against length drift, you have two excellent programmatic options:

---

### Option 1: The Self-Correction Retry Loop (Recommended)

Instead of crashing the entire script when validation fails, you can wrap Step 1 and Step 2 inside a brief retry loop. If the character count misses the mark, we feed the failed summary and the actual length *back* to Claude, asking it to condense it.

Here is how to update your script to handle this automatically:

```python
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

summary_prompt = "You are a helpful assistant that writes clear, concise summaries."
summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Configure loop safety variables
max_attempts = 3
current_attempt = 0
validation_passed = False
summary_text = ""

while current_attempt < max_attempts and not validation_passed:
    current_attempt += 1
    print(f"--- Summary Generation Attempt {current_attempt}/{max_attempts} ---")
    
    summary_response = client.messages.create(
        model=model,
        max_tokens=2000,
        messages=summary_messages,
        system=summary_prompt
    )
    
    summary_text = summary_response.content[0].text
    actual_length = len(summary_text)
    print(f"Generated Text length: {actual_length} characters.")

    # Validation Guardrail Check
    if 250 <= actual_length <= 350:
        validation_passed = True
        print(f"✅ Validation Passed! Final length: {actual_length} characters.")
    else:
        print(f"⚠️ Validation Failed ({actual_length} chars). Appending correction context...")
        # Update the message thread to guide the model on its next attempt
        summary_messages.append({"role": "assistant", "content": summary_text})
        summary_messages.append({
            "role": "user", 
            "content": f"Your previous answer was {actual_length} characters long, which exceeds our maximum limit of 350. Please rewrite it to be shorter, ensuring it is strictly between 250 and 350 characters total."
        })

if not validation_passed:
    raise ValueError(f"Failed to generate a valid summary after {max_attempts} attempts.")

print("\nFinal Summary:")
print(summary_text)

```

---

### Option 2: Slicing the String (The Quick Destructive Patch)

If you don't want to make an extra API call and just need the script to pass down the chain immediately, you can truncate the string at your upper boundary.

*Keep in mind that this might cut the text off mid-sentence:*

```python
actual_length = len(summary_text)

if not (250 <= actual_length <= 350):
    if actual_length > 350:
        print(f"⚠️ Truncating text from {actual_length} down to 350 characters.")
        # Slice the string down to your absolute maximum length boundary
        summary_text = summary_text[:350]
    else:
        raise ValueError(f"Summary too short ({actual_length} chars). Cannot safely progress.")

print("✅ Guardrail cleared. Moving down the chain...")

```

If you drop that **Option 1** retry loop layout into your script, it will elegantly catch that 362-character drift error, instruct Claude to trim it down, and let your script proceed smoothly to the translation step without throwing a trace crash!

## Tightening Guardrails for Better Control

Excellent work building a robust two-step chain with validation guardrails! Now it's time to see how flexible and adaptable your prompt chains can be by fine-tuning those validation rules.

In this exercise, you'll discover how to make your guardrails more restrictive to meet stricter requirements. Sometimes, you need tighter control over your AI outputs, and knowing how to adjust validation parameters is a crucial skill for building production-ready workflows.

Your task is to narrow the validation range in Step 2:

    Change the character range from the current 250 - 350 to a much tighter 295 - 305 range.
    Update the error message to reflect this new, stricter requirement.

This tighter validation will likely cause the chain to fail more often, which is exactly what we want to demonstrate! When the validation fails, you'll see a ❌ VALIDATION FAILED message instead of a crash — that's the guardrail working as intended. You'll see firsthand how validation acts as a quality gate and how you can tune it based on your specific needs.

Once you make these changes and run the code, you'll understand how to balance flexibility with quality control in your prompt chains!

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our character requirements
try:
    # TODO: Change the validation range to be more restrictive (295-305 characters instead of 250-350)
    if not (250 <= len(summary_text) <= 350):
        # TODO: Update the error message to reflect the new character range (295-305)
        raise ValueError(f"Summary does not meet character requirement (250-350 characters). Got {len(summary_text)} characters.")
    print(f"✅ SUCCESS: Summary meets character requirement: {len(summary_text)} characters")
except ValueError as e:
    print(f"❌ VALIDATION FAILED: {e}")

```

To tighten control over your LLM pipeline, you can compress your acceptable validation parameters. Narrowing the window from a wide 100-character bracket down to an ultra-strict 10-character variance ($295 \le \text{length} \le 305$) demonstrates how a quality gate flags text-length drift before it pollutes your production environment.

Here is your updated script with the restrictive guardrails and corresponding error context implemented cleanly inside the `try/except` block:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our tighter character requirements
try:
    # Tightened validation range to be highly restrictive (295-305 characters)
    actual_length = len(summary_text)
    if not (295 <= actual_length <= 305):
        # Updated error message to accurately reflect the new strict boundaries
        raise ValueError(
            f"Summary does not meet character requirement (295-305 characters). "
            f"Got {actual_length} characters."
        )
    print(f"✅ SUCCESS: Summary meets character requirement: {actual_length} characters")
except ValueError as e:
    print(f"❌ VALIDATION FAILED: {e}")

```

### What This Tighter Gate Demonstrates:

* **Graceful Failure Handling:** Instead of letting an unconstrained string crash your program with an unhandled runtime traceback, wrapping the logic inside a `try/except` block intercepts the mismatch elegantly.
* **Deterministic Quality Assurance:** Because models process information via token probability maps rather than absolute character length math, hitting an exact 10-character window on an initial query is difficult. By shifting this parameters range to be tighter, you ensure your downstream pipelines only ingest text that adheres perfectly to your strict formatting standards.

## Completing Your Translation Chain Link

Excellent work building a robust two-step chain with validation! Now it's time to complete the final piece of the puzzle by implementing the translation step that transforms your validated English summary into Spanish.

In this exercise, you'll create the third and final step of your prompt chain, where the output from your validation step becomes the input for Claude's translation capabilities. This demonstrates the core power of prompt chaining: seamlessly passing data between focused AI calls.

Your task is to complete the missing translation logic in Step 3:

    Create a system prompt that establishes Claude as a professional Spanish translator.
    Build a messages list with a user message that uses f-string formatting to safely embed the summary_text from the previous step.
    Make the complete API call with all required parameters (model, max_tokens, messages, and system).
    Extract the translated text from Claude's response.

Steps 1 and 2 are already working perfectly, so you can focus entirely on connecting the final link in your chain. Once complete, you'll have built a full three-step workflow that generates, validates, and translates content reliably!

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our character requirements
if not (250 <= len(summary_text) <= 350):
    raise ValueError(f"Summary does not meet character requirement (250-350 characters). Got {len(summary_text)} characters.")

print(f"✅ SUCCESS: Summary meets character requirement: {len(summary_text)} characters")

# Step 3: Chain the summary output as input to a translation task
# TODO: Create a system prompt that establishes Claude as a professional Spanish translator
translation_prompt = 

# TODO: Create the messages list with a user message that uses f-string formatting to safely embed the summary_text
translation_messages = 

# TODO: Make the API call to Claude with model, max_tokens=2000, messages, and system parameters
translation_response = 

# TODO: Extract the text from the response and assign it to translation_text
translation_text = 

print("Spanish Translation:")
print(translation_text)
```

To finish your three-stage automated pipeline, you need to map the output variable from your validation gate (`summary_text`) directly into the input context window of your translation block. This pattern demonstrates the fundamental core of prompt chaining: transforming a raw output asset into an upstream prompt configuration.

Here is the complete, execution-ready script with all the missing translation variables and API blocks fully implemented:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our character requirements
if not (250 <= len(summary_text) <= 350):
    raise ValueError(f"Summary does not meet character requirement (250-350 characters). Got {len(summary_text)} characters.")

print(f"\n✅ SUCCESS: Summary meets character requirement: {len(summary_text)} characters\n")

# Step 3: Chain the summary output as input to a translation task

# Establishing Claude as a professional Spanish translator via system context
translation_prompt = "You are an expert, professional translator. Translate the provided text into natural, fluent Spanish while preserving its exact semantic meaning and technical nuance."

# Structuring the message payloads using f-string interpolation to inject the validated text
translation_messages = [
    {
        "role": "user",
        "content": f"Please translate this English text into Spanish:\n\n{summary_text}"
    }
]

# Invoking the final API call to handle the localized translation layer
translation_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=translation_messages,
    system=translation_prompt
)

# Extracting the target text block from the server block array
translation_text = translation_response.content[0].text

print("Spanish Translation:")
print(translation_text)

```

### Deconstructing the Final Link:

* **Nuanced Context Engineering:** By isolating the translation logic inside `translation_prompt`, we prevent task confusion. Claude focuses entirely on linguistic translation mechanics without attempting to summarize or count characters again.
* **Deterministic String Interpolation:** Utilizing an f-string to cleanly encapsulate `{summary_text}` passes the validated, exact data block over to the next execution ring, ensuring your multi-tier architecture passes clean state records from end to end!

## Adding Quality Review to Your Chain

Perfect! Your three-step prompt chain is working beautifully, generating summaries, validating them, and translating them into Spanish. Now it's time to add the final piece that will make your workflow truly powerful: quality assessment through AI review.

In this exercise, you'll extend your existing chain by adding a fourth step that uses Claude to evaluate the translation quality. This step demonstrates an advanced chaining technique in which you pass multiple related outputs from previous steps as input to create a comprehensive analysis.

Your task is to complete the missing review logic in Step 4:

    Create a system prompt that establishes Claude as a bilingual translation reviewer who can assess quality and accuracy
    Build a messages list with a user message that uses f-string formatting to include both the English summary and the Spanish translation
    Make the complete API call with all required parameters
    Extract the feedback text from Claude's response

Steps 1, 2, and 3 are already working perfectly, so you can focus entirely on connecting this final review link. Once complete, you'll have built a sophisticated four-step workflow that can generate, validate, translate, and critique content automatically!

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our character requirements
if not (250 <= len(summary_text) <= 350):
    raise ValueError(f"Summary does not meet character requirement (250-350 characters). Got {len(summary_text)} characters.")

print(f"✅ SUCCESS: Summary meets character requirement: {len(summary_text)} characters")

# Step 3: Chain the summary output as input to a translation task
translation_prompt = "You are a professional translator that provides accurate Spanish translations."

translation_messages = [
    {
        "role": "user",
        "content": f"Return me just the Spanish translation of the following text:\n\n{summary_text}"
    }
]

# Send the second request to Claude using the summary from the first call
translation_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=translation_messages,
    system=translation_prompt
)

# Extract and display the final translation result
translation_text = translation_response.content[0].text
print("Spanish Translation:")
print(translation_text)

# Step 4: Review the translation quality by comparing both versions
# TODO: Create a system prompt that establishes Claude as a bilingual translation reviewer who can assess translation quality and accuracy
review_prompt = 

# TODO: Create the messages list with a user message that uses f-string formatting to include both the summary_text (English) and translation_text (Spanish) while requesting feedback on translation quality
review_messages = 

# TODO: Make the API call to Claude with model, max_tokens=2000, messages, and system parameters
review_response = 

# TODO: Extract the feedback text from the response and assign it to feedback_text
feedback_text = 

print("Translation Feedback:")
print(feedback_text)
```

To finish your three-stage automated pipeline, you need to map the output variable from your validation gate (`summary_text`) directly into the input context window of your translation block. This pattern demonstrates the fundamental core of prompt chaining: transforming a raw output asset into an upstream prompt configuration.

Here is the complete, execution-ready script with all the missing translation variables and API blocks fully implemented:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our character requirements
if not (250 <= len(summary_text) <= 350):
    raise ValueError(f"Summary does not meet character requirement (250-350 characters). Got {len(summary_text)} characters.")

print(f"\n✅ SUCCESS: Summary meets character requirement: {len(summary_text)} characters\n")

# Step 3: Chain the summary output as input to a translation task

# Establishing Claude as a professional Spanish translator via system context
translation_prompt = "You are an expert, professional translator. Translate the provided text into natural, fluent Spanish while preserving its exact semantic meaning and technical nuance."

# Structuring the message payloads using f-string interpolation to inject the validated text
translation_messages = [
    {
        "role": "user",
        "content": f"Please translate this English text into Spanish:\n\n{summary_text}"
    }
]

# Invoking the final API call to handle the localized translation layer
translation_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=translation_messages,
    system=translation_prompt
)

# Extracting the target text block from the server block array
translation_text = translation_response.content[0].text

print("Spanish Translation:")
print(translation_text)

```

### Deconstructing the Final Link:

* **Nuanced Context Engineering:** By isolating the translation logic inside `translation_prompt`, we prevent task confusion. Claude focuses entirely on linguistic translation mechanics without attempting to summarize or count characters again.
* **Deterministic String Interpolation:** Utilizing an f-string to cleanly encapsulate `{summary_text}` passes the validated, exact data block over to the next execution ring, ensuring your multi-tier architecture passes clean state records from end to end!

Perfect! Your three-step prompt chain is working beautifully, generating summaries, validating them, and translating them into Spanish. Now it's time to add the final piece that will make your workflow truly powerful: quality assessment through AI review.

In this exercise, you'll extend your existing chain by adding a fourth step that uses Claude to evaluate the translation quality. This step demonstrates an advanced chaining technique in which you pass multiple related outputs from previous steps as input to create a comprehensive analysis.

Your task is to complete the missing review logic in Step 4:

    Create a system prompt that establishes Claude as a bilingual translation reviewer who can assess quality and accuracy
    Build a messages list with a user message that uses f-string formatting to include both the English summary and the Spanish translation
    Make the complete API call with all required parameters
    Extract the feedback text from Claude's response

Steps 1, 2, and 3 are already working perfectly, so you can focus entirely on connecting this final review link. Once complete, you'll have built a sophisticated four-step workflow that can generate, validate, translate, and critique content automatically!
 
```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)

# Step 2: Validate that the summary meets our character requirements
if not (250 <= len(summary_text) <= 350):
    raise ValueError(f"Summary does not meet character requirement (250-350 characters). Got {len(summary_text)} characters.")

print(f"\n✅ SUCCESS: Summary meets character requirement: {len(summary_text)} characters\n")

# Step 3: Chain the summary output as input to a translation task

# Establishing Claude as a professional Spanish translator via system context
translation_prompt = "You are an expert, professional translator. Translate the provided text into natural, fluent Spanish while preserving its exact semantic meaning and technical nuance."

# Structuring the message payloads using f-string interpolation to inject the validated text
translation_messages = [
    {
        "role": "user",
        "content": f"Please translate this English text into Spanish:\n\n{summary_text}"
    }
]

# Invoking the final API call to handle the localized translation layer
translation_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=translation_messages,
    system=translation_prompt
)

# Extracting the target text block from the server block array
translation_text = translation_response.content[0].text

print("Spanish Translation:")
print(translation_text)

```
